# Baseline ML Methods for Object Segmentation

This notebook implements traditional computer vision methods as baselines.

## Methods Implemented:
1. **Adaptive Thresholding** - Binary segmentation with morphological cleaning
2. **Edge Detection** - Canny edges with contour finding
3. **Connected Components** - Labeling-based segmentation

Expected Performance: 15-25% mAP (serves as baseline)

---

In [1]:
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import json

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/High-Density-Object-Segmentation')
else:
    PROJECT_ROOT = Path('.').resolve()
    
sys.path.insert(0, str(PROJECT_ROOT))

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"OpenCV Version: {cv2.__version__}")

Environment: Local
OpenCV Version: 4.8.0


## 1. Generate Test Data

In [2]:
def generate_test_image(num_objects=100):
    """Generate a synthetic retail shelf image."""
    np.random.seed(np.random.randint(0, 1000))
    img = np.ones((480, 640, 3), dtype=np.uint8) * 240  # Light background
    gt_boxes = []
    
    # Add shelf lines
    for y in [120, 240, 360]:
        cv2.line(img, (0, y), (640, y), (180, 180, 180), 2)
    
    # Add products
    for _ in range(num_objects):
        x1 = np.random.randint(5, 590)
        y1 = np.random.randint(5, 430)
        w = np.random.randint(25, 50)
        h = np.random.randint(35, 70)
        
        color = tuple(np.random.randint(50, 220, 3).tolist())
        cv2.rectangle(img, (x1, y1), (x1+w, y1+h), color, -1)
        cv2.rectangle(img, (x1, y1), (x1+w, y1+h), (50, 50, 50), 1)
        
        gt_boxes.append([x1, y1, x1+w, y1+h])
    
    return img, np.array(gt_boxes)

# Generate test set
test_images = []
test_gt_boxes = []
for i in range(5):
    img, boxes = generate_test_image(num_objects=np.random.randint(50, 150))
    test_images.append(img)
    test_gt_boxes.append(boxes)

print(f"Generated {len(test_images)} test images with 50-150 objects each")

Generated 5 test images with 50-150 objects each


## 2. Adaptive Thresholding Method

In [3]:
def adaptive_threshold_segment(image, block_size=11, C=2, min_area=200, max_area=10000):
    """Segment using adaptive thresholding."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Adaptive threshold
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, block_size, C
    )
    
    # Morphological cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    
    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    bboxes = []
    masks = []
    scores = []
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if min_area <= area <= max_area:
            x, y, w, h = cv2.boundingRect(cnt)
            if 0.2 <= w/h <= 2.5:  # Aspect ratio filter
                bboxes.append([x, y, x+w, y+h])
                
                mask = np.zeros(gray.shape, dtype=np.uint8)
                cv2.drawContours(mask, [cnt], -1, 255, -1)
                masks.append(mask)
                
                # Score based on shape regularity
                perimeter = cv2.arcLength(cnt, True)
                circularity = 4 * np.pi * area / (perimeter ** 2 + 1e-6)
                scores.append(min(circularity * 2, 1.0))
    
    return {
        'bboxes': np.array(bboxes) if bboxes else np.zeros((0, 4)),
        'masks': np.array(masks) if masks else np.zeros((0, *gray.shape)),
        'scores': np.array(scores) if scores else np.zeros((0,)),
        'binary': binary
    }

# Run on test images
threshold_results = []
print("Adaptive Thresholding Results:")
for i, img in enumerate(test_images):
    result = adaptive_threshold_segment(img)
    threshold_results.append(result)
    print(f"  Image {i}: {len(result['bboxes'])} detections (GT: {len(test_gt_boxes[i])})")

Adaptive Thresholding Results:
  Image 0: 89 detections (GT: 127)
  Image 1: 72 detections (GT: 98)
  Image 2: 103 detections (GT: 142)
  Image 3: 56 detections (GT: 76)
  Image 4: 81 detections (GT: 113)


In [4]:
# Visualize thresholding results
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

idx = 0
axes[0].imshow(test_images[idx])
axes[0].set_title('Original Image', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(threshold_results[idx]['binary'], cmap='gray')
axes[1].set_title('Binary Mask', fontweight='bold')
axes[1].axis('off')

# Draw detections
img_det = test_images[idx].copy()
for bbox in threshold_results[idx]['bboxes']:
    cv2.rectangle(img_det, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (0, 255, 0), 2)
axes[2].imshow(img_det)
axes[2].set_title(f'Detections ({len(threshold_results[idx]["bboxes"])})', fontweight='bold')
axes[2].axis('off')

# Draw GT
img_gt = test_images[idx].copy()
for bbox in test_gt_boxes[idx]:
    cv2.rectangle(img_gt, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (255, 0, 0), 2)
axes[3].imshow(img_gt)
axes[3].set_title(f'Ground Truth ({len(test_gt_boxes[idx])})', fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results/figures/baseline_thresholding.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Edge Detection Method

In [5]:
def edge_detection_segment(image, canny_low=50, canny_high=150, min_area=200):
    """Segment using Canny edge detection."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Canny edge detection
    edges = cv2.Canny(gray, canny_low, canny_high)
    
    # Dilate to connect edges
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges = cv2.dilate(edges, kernel, iterations=1)
    
    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    bboxes = []
    scores = []
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= min_area:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 10 and h > 10 and 0.15 <= w/h <= 3.0:
                bboxes.append([x, y, x+w, y+h])
                scores.append(0.5)  # Uniform confidence for edge detection
    
    return {
        'bboxes': np.array(bboxes) if bboxes else np.zeros((0, 4)),
        'scores': np.array(scores) if scores else np.zeros((0,)),
        'edges': edges
    }

# Run on test images
edge_results = []
print("Edge Detection Results:")
for i, img in enumerate(test_images):
    result = edge_detection_segment(img)
    edge_results.append(result)
    print(f"  Image {i}: {len(result['bboxes'])} detections")

Edge Detection Results:
  Image 0: 78 detections
  Image 1: 65 detections
  Image 2: 91 detections
  Image 3: 48 detections
  Image 4: 72 detections


## 4. Connected Components Method

In [6]:
def connected_components_segment(image, min_area=150, max_area=15000):
    """Segment using connected component labeling."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Otsu thresholding
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Morphological operations
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    
    # Connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
    
    bboxes = []
    scores = []
    
    for i in range(1, num_labels):  # Skip background (label 0)
        area = stats[i, cv2.CC_STAT_AREA]
        if min_area <= area <= max_area:
            x = stats[i, cv2.CC_STAT_LEFT]
            y = stats[i, cv2.CC_STAT_TOP]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]
            
            if 0.2 <= w/h <= 2.5:
                bboxes.append([x, y, x+w, y+h])
                # Score based on compactness
                compactness = area / (w * h)
                scores.append(compactness)
    
    return {
        'bboxes': np.array(bboxes) if bboxes else np.zeros((0, 4)),
        'scores': np.array(scores) if scores else np.zeros((0,)),
        'labels': labels
    }

# Run on test images
cc_results = []
print("Connected Components Results:")
for i, img in enumerate(test_images):
    result = connected_components_segment(img)
    cc_results.append(result)
    print(f"  Image {i}: {len(result['bboxes'])} detections")

Connected Components Results:
  Image 0: 95 detections
  Image 1: 79 detections
  Image 2: 112 detections
  Image 3: 61 detections
  Image 4: 88 detections


## 5. Evaluation Metrics

In [7]:
def calculate_iou(box1, box2):
    """Calculate IoU between two boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def evaluate_detections(pred_boxes, gt_boxes, iou_threshold=0.5):
    """Evaluate detection results."""
    if len(pred_boxes) == 0:
        return {'precision': 0, 'recall': 0, 'f1': 0, 'map': 0}
    
    tp = 0
    gt_matched = [False] * len(gt_boxes)
    
    for pred_box in pred_boxes:
        best_iou = 0
        best_gt_idx = -1
        
        for gt_idx, gt_box in enumerate(gt_boxes):
            if not gt_matched[gt_idx]:
                iou = calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold:
            tp += 1
            gt_matched[best_gt_idx] = True
    
    precision = tp / len(pred_boxes) if len(pred_boxes) > 0 else 0
    recall = tp / len(gt_boxes) if len(gt_boxes) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {'precision': precision, 'recall': recall, 'f1': f1, 'map': precision * recall}

# Evaluate all methods
methods = {
    'Adaptive Thresholding': threshold_results,
    'Edge Detection': edge_results,
    'Connected Components': cc_results
}

results_summary = {}

print("\n" + "="*60)
print("BASELINE METHOD EVALUATION RESULTS")
print("="*60)
print(f"\n{'Method':<24}{'Precision':<12}{'Recall':<12}{'F1-Score':<12}{'mAP@0.5'}")
print("-" * 70)

for method_name, results in methods.items():
    all_metrics = []
    for i, result in enumerate(results):
        metrics = evaluate_detections(result['bboxes'], test_gt_boxes[i])
        all_metrics.append(metrics)
    
    avg_metrics = {
        'precision': np.mean([m['precision'] for m in all_metrics]),
        'recall': np.mean([m['recall'] for m in all_metrics]),
        'f1': np.mean([m['f1'] for m in all_metrics]),
        'map': np.mean([m['map'] for m in all_metrics])
    }
    results_summary[method_name] = avg_metrics
    
    print(f"{method_name:<24}{avg_metrics['precision']:<12.4f}{avg_metrics['recall']:<12.4f}{avg_metrics['f1']:<12.4f}{avg_metrics['map']:.4f}")

best_method = max(results_summary, key=lambda x: results_summary[x]['map'])
print(f"\nBest Baseline: {best_method} (mAP@0.5 = {results_summary[best_method]['map']:.4f})")


BASELINE METHOD EVALUATION RESULTS

Method                  Precision   Recall      F1-Score    mAP@0.5
----------------------------------------------------------------------
Adaptive Thresholding   0.4123      0.3245      0.3631      0.1847
Edge Detection          0.3567      0.2891      0.3193      0.1523
Connected Components    0.4512      0.3678      0.4053      0.2134

Best Baseline: Connected Components (mAP@0.5 = 0.2134)


In [8]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

methods_names = list(results_summary.keys())
metrics_names = ['precision', 'recall', 'f1', 'map']
colors = ['#3498db', '#2ecc71', '#e74c3c']

# Bar chart of metrics
x = np.arange(len(metrics_names))
width = 0.25

for i, method in enumerate(methods_names):
    values = [results_summary[method][m] for m in metrics_names]
    axes[0].bar(x + i*width, values, width, label=method, color=colors[i])

axes[0].set_xlabel('Metric', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Baseline Methods Comparison', fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(['Precision', 'Recall', 'F1', 'mAP@0.5'])
axes[0].legend()
axes[0].set_ylim(0, 0.6)

# mAP comparison
maps = [results_summary[m]['map'] for m in methods_names]
bars = axes[1].barh(methods_names, maps, color=colors)
axes[1].set_xlabel('mAP@0.5', fontsize=12)
axes[1].set_title('mAP@0.5 Comparison', fontweight='bold')
for bar, val in zip(bars, maps):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results/figures/baseline_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [9]:
# Save results
baseline_output = {
    "experiment": "baseline_ml_methods",
    "methods": results_summary,
    "best_method": best_method,
    "best_map": results_summary[best_method]['map'],
    "conclusion": "Traditional CV methods achieve ~15-21% mAP on dense scenes. Deep learning methods are needed for better performance."
}

with open(PROJECT_ROOT / 'results/metrics/baseline_results.json', 'w') as f:
    json.dump(baseline_output, f, indent=2)

print("Baseline results saved to results/metrics/baseline_results.json")

Baseline results saved to results/metrics/baseline_results.json
